In [1]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [2]:

root_dir = r'Y:\ZHL\isds\PS\task0718'
merge_dir = os.path.join(root_dir, 'merge_dir')
root_folder_id = '1M80wzVS77_RBMqfelX6C4tP96ZVFbMG3'
client_secret = r"E:\data\202502_signboard\data_annotation\docs\client_secret.json"
token_path = 'token.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

slam_root_folder_id = '11oS7ftYlCPMI8yckSTBY_fkYDCtdBcY2'

In [3]:
import os
import io
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request


def authenticate_with_google(token_path, client_secret_path):
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token_file:
            token_file.write(creds.to_json())

    service = build('drive', 'v3', credentials=creds)
    return service


def download_large_file(service, file_id, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if os.path.exists(file_path):
        print(f"⚠️ 已存在，跳过: {file_path}")
        return
    print(f"⬇️ Downloading {file_path}")
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(file_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"⬇️ Downloading {file_path}: {int(status.progress() * 100)}%")
    print(f"✅ Finished: {file_path}")

def download_folder_recursive(service, folder_id, save_path):
    os.makedirs(save_path, exist_ok=True)
    query = f"'{folder_id}' in parents and trashed = false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    for item in items:
        file_id = item['id']
        file_name = item['name']
        file_mime = item['mimeType']
        full_path = os.path.join(save_path, file_name)

        if file_mime == 'application/vnd.google-apps.folder':
            download_folder_recursive(service, file_id, full_path)
        else:
            download_large_file(service, file_id, full_path)

def download_subfolder_task(folder_obj, root_save_path, token_path, client_secret_path):
    # 每个线程都单独认证，避免多线程共享service导致问题
    service = authenticate_with_google(token_path, client_secret_path)
    folder_id = folder_obj['id']
    folder_name = folder_obj['name']
    target_path = os.path.join(root_save_path, folder_name)
    print(f"\n📁 Starting folder: {folder_name}")
    download_folder_recursive(service, folder_id, target_path)

def download_all_subfolders_parallel(token_path, client_secret_path, root_folder_id, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # 主线程先获取子文件夹列表
    service = authenticate_with_google(token_path, client_secret_path)
    query = f"'{root_folder_id}' in parents and trashed = false and mimeType = 'application/vnd.google-apps.folder'"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    folders = results.get('files', [])

    print(f"将并发下载 {len(folders)} 个子文件夹...\n")

    with ThreadPoolExecutor(max_workers=len(folders)) as executor:
        for folder in folders:
            executor.submit(download_subfolder_task, folder, save_dir, token_path, client_secret_path)



In [ ]:

# download_all_subfolders_parallel(token_path, client_secret, root_folder_id, root_dir)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=690434021289-qg1r4ut91uq2pn2oj6tei9bsvqtmv52n.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A58991%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=tCp0x244PyD9qCJFifSSXMrKqAILzc&access_type=offline
将并发下载 6 个子文件夹...


📁 Starting folder: 11-46-14

📁 Starting folder: 12-26-26

📁 Starting folder: 16-08-14

📁 Starting folder: 14-17-25

📁 Starting folder: 15-31-17

📁 Starting folder: 11-23-49
⚠️ 已存在，跳过: Y:\ZHL\isds\PS\task0718\12-26-26\rectified_images.zip
⚠️ 已存在，跳过: Y:\ZHL\isds\PS\task0718\15-31-17\rectified_images.zip
⚠️ 已存在，跳过: Y:\ZHL\isds\PS\task0718\12-26-26\imu_data_20250718122624968_20250718123601815.txt
⚠️ 已存在，跳过: Y:\ZHL\isds\PS\task0718\15-31-17\imu_data_20250718153116662_20250718154122619.txt
⚠️ 已存在，跳过: Y:\ZHL\isds\PS\task0718\12-26-26\gps_data_20250718122624800_20250718123601600.txt
⚠️ 已存在，跳过: Y:\ZHL\isds\PS\task0718\

In [11]:
def uzip_dirs(root_dir):
    sub_dir_list = os.listdir(root_dir)
    for sub_dir_name in sub_dir_list:
        sub_dir = os.path.join(root_dir, sub_dir_name)
        zip_path = os.path.join(sub_dir, 'rectified_images.zip')
        if not os.path.exists(zip_path):
            print(f'{zip_path} not exists')
        else:
            print(f'{zip_path} unzip...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(zip_path.replace('.zip', ''))
            print(f'{zip_path} done\n')

In [ ]:
# uzip_dirs(root_dir)

Y:\ZHL\isds\PS\task0718\11-23-49\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0718\11-23-49\rectified_images.zip done

Y:\ZHL\isds\PS\task0718\11-46-14\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0718\11-46-14\rectified_images.zip done

Y:\ZHL\isds\PS\task0718\12-26-26\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0718\12-26-26\rectified_images.zip done

Y:\ZHL\isds\PS\task0718\14-17-25\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0718\14-17-25\rectified_images.zip done

Y:\ZHL\isds\PS\task0718\15-31-17\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0718\15-31-17\rectified_images.zip done

Y:\ZHL\isds\PS\task0718\16-08-14\rectified_images.zip unzip...
Y:\ZHL\isds\PS\task0718\16-08-14\rectified_images.zip done



In [29]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or not sub_name.startswith('1'):
            continue
        cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'rectified_images', 'rectified_images', cam_name)
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=30)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter' 
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [30]:
process_dirs(root_dir)

Y:\ZHL\isds\PS\task0718\11-23-49\rectified_images\rectified_images\cam_DA4930148 selecting...


  0%|          | 0/343 [00:00<?, ?it/s]

100%|██████████| 343/343 [00:05<00:00, 62.41it/s]


Y:\ZHL\isds\PS\task0718\11-23-49\rectified_images\rectified_images\cam_DA4930148_select filtering...


100%|██████████| 343/343 [00:28<00:00, 12.15it/s]


...

Copied: cam_image_20250718160828300.jpg (Max similarity: 0.36)
Copied: cam_image_20250718160831299.jpg (Max similarity: 0.45)
Copied: cam_image_20250718160834300.jpg (Max similarity: 0.45)
Copied: cam_image_20250718160837299.jpg (Max similarity: 0.38)
Copied: cam_image_20250718160840300.jpg (Max similarity: 0.38)
Copied: cam_image_20250718160843300.jpg (Max similarity: 0.35)
Copied: cam_image_20250718160846299.jpg (Max similarity: 0.39)
Copied: cam_image_20250718160849300.jpg (Max similarity: 0.39)
Copied: cam_image_20250718160852400.jpg (Max similarity: 0.37)
Copied: cam_image_20250718160855400.jpg (Max similarity: 0.32)
Copied: cam_image_20250718160858400.jpg (Max similarity: 0.38)
Copied: cam_image_20250718160901500.jpg (Max similarity: 0.38)
Copied: cam_image_20250718160904500.jpg (Max similarity: 0.35)
Copied: cam_image_20250718160907499.jpg (Max similarity: 0.40)
Copied: cam_image_20250718161019600.jpg (Max similarity: 0.37)
Copied: cam_image_20250718161022599.jpg (Max similarity

100%|██████████| 201/201 [00:06<00:00, 30.40it/s]


Y:\ZHL\isds\PS\task0718\16-08-14\rectified_images\rectified_images\cam_DA6102933_select filtering...


100%|██████████| 201/201 [00:15<00:00, 12.65it/s]


Copied: cam_image_20250718160828300.jpg (Max similarity: 0.34)
Copied: cam_image_20250718160837299.jpg (Max similarity: 0.30)
Copied: cam_image_20250718160840300.jpg (Max similarity: 0.30)
Copied: cam_image_20250718160843300.jpg (Max similarity: 0.21)
Copied: cam_image_20250718160846299.jpg (Max similarity: 0.19)
Copied: cam_image_20250718160849300.jpg (Max similarity: 0.24)
Copied: cam_image_20250718160852400.jpg (Max similarity: 0.23)
Copied: cam_image_20250718160855400.jpg (Max similarity: 0.17)
Copied: cam_image_20250718160858400.jpg (Max similarity: 0.20)
Copied: cam_image_20250718160901500.jpg (Max similarity: 0.22)
Copied: cam_image_20250718160904500.jpg (Max similarity: 0.22)
Copied: cam_image_20250718160907499.jpg (Max similarity: 0.23)
Copied: cam_image_20250718161019500.jpg (Max similarity: 0.18)
Copied: cam_image_20250718161022500.jpg (Max similarity: 0.18)
Copied: cam_image_20250718161025499.jpg (Max similarity: 0.18)
Copied: cam_image_20250718161028499.jpg (Max similarity

In [31]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir):
            continue
        cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'rectified_images', 'rectified_images', cam_name+'_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)
    


In [32]:
img_merge(root_dir, merge_dir)

100%|██████████| 232/232 [00:05<00:00, 41.25it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 127/127 [00:04<00:00, 30.38it/s]


In [33]:
print(len(os.listdir(merge_dir)))

3385


In [34]:
import zipfile
import os

def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

zip_folder_to_path(
    source_folder=merge_dir,
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'.zip')
)

zip 'Y:\ZHL\isds\PS\task0718\merge_dir' to 'Y:\ZHL\isds\PS\task0718\task0718.zip'


In [4]:
download_all_subfolders_parallel(token_path, client_secret, slam_root_folder_id, root_dir)

将并发下载 6 个子文件夹...


📁 Starting folder: 11-23-49

📁 Starting folder: 16-08-14

📁 Starting folder: 11-46-14

📁 Starting folder: 14-17-25

📁 Starting folder: 15-31-17

📁 Starting folder: 12-26-26
⬇️ Downloading Y:\ZHL\isds\PS\task0718\12-26-26\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\15-31-17\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\14-17-25\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\16-08-14\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\11-46-14\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\11-23-49\segment3\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\14-17-25\poses_tum_converted.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0718\14-17-25\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0718\14-17-25\14-17-25.pcd
⬇️ Downloading Y:\ZHL\isds\PS\task0718\16-08-14\poses_tum_converted.txt: 100%
⬇️ Downloading Y:\ZHL\isds\PS\task0718\11-46-14\poses_tum_converted.txt: 100%
✅ F